# 03 — Cache the frozen body's features, once

**Gate this notebook closes: G3.**

This is the notebook that makes the project feel small. The ResNet18 body never changes
during training, so its 512-vector per tile can be computed once. After that, fitting the
three-class head takes **seconds** — which turns the class definitions, the loss weights, the
input polarity and the whole of Approach 3 into experiments you run in a coffee break rather
than decisions you have to defend in advance.

~1.5 hours on CPU for ~14,000 tiles. Run it once per *initialisation* (this notebook does
ImageNet; approach 3's notebook 02 does SimCLR against the same manifest) and once more per
polarity if you run ablation A2.

**Augmentation is off here, and that is not an oversight.** These features must be the
features step 8 will compute at inference, which is the `augment=False` path — the identical
transform, no jitter, no flips. The augmented path is used in notebook 05 only, where the
body is being fine-tuned and cached features no longer apply.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

import backend_path            # noqa: F401 - puts the demo backend on sys.path
backend_path.ensure_dirs()
print("backend :", backend_path.BACKEND_ROOT)
print("tiles   :", backend_path.TILES_DIR)
print("models  :", backend_path.MODELS_DIR)

In [ ]:
INIT = "imagenet"        # "imagenet" here; approach 3 runs the same cell with "simclr"
INVERT_POLARITY = False  # ablation A2 flips this. Recorded in the cache and the manifest.
BATCH = 64               # CPU: large enough to amortise the per-call overhead, small
                         # enough not to page. 64 x 3 x 224 x 224 float32 is ~38 MB.

print(f"initialisation : {INIT}")
print(f"polarity       : {'nuclei dark (inverted)' if INVERT_POLARITY else 'nuclei bright'}")

## Vendor the weights

Downloaded once, copied into `models/tissue_type/pretrained/`, and its SHA-256 recorded.
Nothing in this pipeline is fetched from a model hub at run time — not at training and
certainly not at inference, where a silently updated checkpoint would change a clinical
number without a commit.

In [ ]:
import models

if INIT == "imagenet":
    path = models.vendor_imagenet_weights(backend_path.PRETRAINED_DIR)
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")
    print(f"  sha256 {models.sha256_of(path)}")

net = models.resnet18_backbone(INIT, weights_dir=backend_path.PRETRAINED_DIR)
body = models.feature_extractor(net)

trainable = sum(p.numel() for p in body.parameters() if p.requires_grad)
total = sum(p.numel() for p in body.parameters())
print(f"\nbody: {total:,} parameters, {trainable:,} trainable (frozen, so zero)")

# The norm of conv1 is the fingerprint that tells two initialisations apart. Approach 3
# asserts these differ - a silent partial load would otherwise be invisible.
import torch
print(f"conv1 weight norm: {float(torch.linalg.norm(net.conv1.weight)):.4f}")

## Extract

In [ ]:
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

import datasets

rows = datasets.read_manifest(backend_path.TILES_DIR / "tiles_manifest.csv")
fingerprint = datasets.manifest_fingerprint(rows)
print(f"{len(rows):,} tiles, manifest fingerprint {fingerprint[:16]}")

dataset = datasets.TileDataset(rows, backend_path.TILES_DIR,
                               augment=False, invert=INVERT_POLARITY)
# num_workers=0 deliberately: on Windows each worker re-imports the module tree, and
# reading a 15 KB PNG is not the bottleneck - the forward pass is.
loader = DataLoader(dataset, batch_size=BATCH, shuffle=False, num_workers=0)

features = np.zeros((len(dataset), models.FEATURE_DIM), dtype=np.float32)
labels = np.zeros(len(dataset), dtype=np.int64)

cursor = 0
with torch.inference_mode():
    for batch, batch_labels in tqdm(loader, desc="frozen body"):
        vectors = body(batch).numpy()
        features[cursor:cursor + len(vectors)] = vectors
        labels[cursor:cursor + len(vectors)] = batch_labels.numpy()
        cursor += len(vectors)

assert cursor == len(dataset), f"extracted {cursor} of {len(dataset)} tiles"
print(f"\nfeatures: {features.shape}, {features.nbytes / 1e6:.1f} MB")

In [ ]:
cache = backend_path.FEATURES_DIR / f"{INIT}{'_inverted' if INVERT_POLARITY else ''}.npz"
np.savez_compressed(
    cache,
    features=features,
    labels=labels,
    # The fingerprint is what stops a feature array being paired with a different
    # manifest. That failure trains to about chance and looks exactly like a model
    # that will not learn - the quietest bug in the whole plan.
    fingerprint=np.array(fingerprint),
    init=np.array(INIT),
    invert=np.array(INVERT_POLARITY),
)
print(f"wrote {cache.name} ({cache.stat().st_size / 1e6:.1f} MB)")

## G3 — deterministic, aligned, and carrying signal

Three checks. The third is deliberately optimistic — a random split on tile features will
flatter any model — because its only job is to prove the features are not noise. The honest
number comes from notebook 04.

In [ ]:
# 1. Determinism. Eval mode, no dropout, no augmentation: the same tile must give
#    the same vector. If not, the cache cannot be trusted against the manifest.
subset = datasets.TileDataset(rows[:100], backend_path.TILES_DIR,
                              augment=False, invert=INVERT_POLARITY)
with torch.inference_mode():
    again = body(torch.stack([subset[i][0] for i in range(100)])).numpy()
assert np.allclose(again, features[:100], atol=1e-5), "feature extraction is not deterministic"
print("G3a OK - re-extraction reproduces the first 100 vectors")

# 2. Alignment. Row order is the manifest's order, asserted by the fingerprint.
stored = np.load(cache, allow_pickle=False)
assert str(stored["fingerprint"]) == fingerprint
assert np.array_equal(stored["labels"], labels)
print("G3b OK - the cache carries the manifest fingerprint it was built from")

In [ ]:
# 3. Sanity probe. Logistic regression on the cached features, RANDOM split - which
#    is exactly the split the plan forbids for reporting. That is fine here: the
#    question is only "do these features carry signal at all", and a number that is
#    known to be optimistic answers it without pretending to be a result.
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    features, labels, test_size=0.25, random_state=0, stratify=labels
)
probe = LogisticRegression(max_iter=2000, class_weight="balanced").fit(x_train, y_train)
accuracy = probe.score(x_test, y_test)

print(f"probe accuracy (RANDOM split - optimistic by design): {accuracy:.3f}")
assert accuracy > 0.70, (
    f"{accuracy:.3f} is too low even for a flattering split. The features are not "
    "carrying class information - check the input transform and gate G2's eyeball panel."
)
print("\nG3c OK - the features carry signal. The honest number is notebook 04's.")

---

- [ ] G3a — extraction is deterministic
- [ ] G3b — the cache is fingerprinted against its manifest
- [ ] G3c — the probe clears ~0.75 on a (deliberately optimistic) random split

Next: notebook 04, where the split stops being random and the number starts meaning
something.